# SCoNE usage example

In [16]:
import numpy as np
import pandas as pd
import os
from algorithms.SCoNE import SCoNE_parallel

cwd = os.getcwd()
parent_dir = os.path.dirname(os.getcwd())

G = pd.read_csv(f'{parent_dir}/example_data/G.csv', index_col=0).to_numpy(dtype=float)
C = pd.read_csv(f'{parent_dir}/example_data/C.csv', index_col=0).to_numpy(dtype=float)
Z = pd.read_csv(f'{parent_dir}/example_data/Z.csv', index_col=0).to_numpy(dtype=float)

factor_matrices, loss_function = SCoNE_parallel(
    G, 
    C, 
    Z, 
    rank=3,
    alpha=max(G.max(),C.max())**2,
    lambda_H_G=1e-4, 
    lambda_H_C=1e-4, 
    lambda_Gloss=1,  
    num_init=3,
    init='nndsvda',
    G_loss_type='kl_div', 
    C_loss_type='kl_div', 
)

In [20]:
loss_function.keys()

dict_keys(['total_loss', 'G_loss', 'C_loss', 'G_plus_C_loss', 'l2_regularization', 'l1_regularization', 'W_norm', 'H_G_norm', 'U_G_norm', 'H_C_norm', 'U_C_norm', 'W_sparsity', 'H_G_sparsity', 'U_G_sparsity', 'H_C_sparsity', 'U_C_sparsity', 'coph_corr'])

# Demo implementation & comparison of all algorithms

In [1]:
# import necessary libraries
import numpy as np
from simulate_data import simulate_views

## Simulate data

In [2]:
matrices = simulate_views(n=1000, num_genes=10, M_C=10, M_Z=3, rank=3, seed=0)
print(matrices.keys())

# make sure dtype is float
G = matrices['G'].astype(float)
C = matrices['C'].astype(float)
Z = matrices['Z'].astype(float)

dict_keys(['G', 'C', 'Z', 'W_C', 'W_G', 'H_G', 'H_C', 'U_G', 'U_C'])


## Run SCoNE

In [3]:
from algorithms.SCoNE import SCoNE_parallel

factor_matrices, loss_function = SCoNE_parallel(
    G, C, Z, rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=1e-4, lambda_H_C=1e-4, lambda_Gloss=1,   # regularization parameters
    num_init=3,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)

Evaluate similarity between simulated, true $W_C$ and estimated $W$ from SCoNE

In [ ]:
from evaluation.reconstruction_evaluation import best_permutation_similarity

similarity, permutation = best_permutation_similarity(matrices['W_C'],factor_matrices['W'])
print(similarity)
print(permutation) # optimal permutation of columns

0.7678109882625739
[2 1 0]
<class 'numpy.ndarray'>


## Run HNMF

In [15]:
factor_matrices, loss_function = SCoNE_parallel(
    G, C, Z=None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=5,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.4552018765389476

## Run HNMF (res)

In [16]:
from algorithms.SCoNE import proj_nonneg

C_resid = proj_nonneg(C - Z @ np.linalg.lstsq(Z, C,rcond=None)[0])
G_resid = proj_nonneg(G - Z @ np.linalg.lstsq(Z, G,rcond=None)[0])

factor_matrices, loss_function = SCoNE_parallel(
    G_resid, C_resid, Z=None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=5,init='nndsvda',G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7016345250240378

## Run C-CoNE

In [19]:
factor_matrices, loss_function = SCoNE_parallel(
    G=None, C=C, Z=Z, rank=3,
    num_init=5,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6314014946888004

## Run G-CoNE

In [20]:
factor_matrices, loss_function = SCoNE_parallel(
   G=G,C=None,Z=Z, rank=3,
    num_init=5,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6881441064824108

## Run C-NMF

In [21]:
factor_matrices, loss_function = SCoNE_parallel(
    G=None, C=C, Z=None, rank=3,
    num_init=5,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.4894738245544161

## Run G-NMF

In [23]:
factor_matrices, loss_function = SCoNE_parallel(
    G=G, C=None, Z=None, rank=3,
    num_init=5,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6493930065217487

## Run RGWAS

In [28]:
import os
import pandas as pd
from algorithms.RGWASWrapper import RGWASWrapper

cwd = os.getcwd()
parent_dir = os.path.dirname(os.getcwd())
pd.DataFrame(matrices["G"]).to_csv(f'{parent_dir}/example_data/G.csv') 
pd.DataFrame(matrices["C"]).to_csv(f'{parent_dir}/example_data/C.csv') 
pd.DataFrame(matrices["Z"]).to_csv(f'{parent_dir}/example_data/Z.csv') 

r_path = "/path/to/Rscript"  # Replace with the path to your Rscript executable

factor_matrices, loss_function = RGWASWrapper(
    r_path=r_path,
    G_path=f'{parent_dir}/example_data/G.csv',
    C_path=f'{parent_dir}/example_data/C.csv',
    Z_path=f'{parent_dir}/example_data/Z.csv',
    rank=5, num_init=5, write_all_init=False)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.3258309962182316

## Run MVBC

In [26]:
import algorithms.MVBCWrapper as MVBCWrapper
import importlib
importlib.reload(MVBCWrapper)

factor_matrices, loss_function = MVBCWrapper.MVBCWrapper(
    G_path=f'{parent_dir}/example_data/G.csv', C_path=f'{parent_dir}/example_data/C.csv', 
    rank=3, lambda_W=1, lambda_H_G=1, lambda_H_C=1,  
    r_path=r_path) # REPLACE WITH CORRECT Rscript path
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.3444022995385595